# TEXT ANALYTICS

## LIBRARIES

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import fasttext
import re

In [2]:
data = pd.read_csv("Dataset/initial_data.csv")

## TEXT PROCESSING

### LYRICS LENGTH

In [3]:
data["words"] = data["Lyric"].str.split().str.len() # splitting based on whitespaces
data

,Lyric,genre,words
0,Feels like Im covered in lies so turn off the ...,Metal,179
1,"Works of art, painted black Magniloquent, blee...",Metal,121
2,Into the cage like an animal You must survive ...,Metal,140
3,Paralysed in pleasure I hear you call Lost my ...,Metal,178
4,"Unseen, cheap, faithful weapon Buried devils a...",Metal,28
...,...,...,...
541713,[Verse 1] When the sun sinks down and dreams s...,Country,249
541714,I watched from the window as she slipped from ...,Country,203
541715,"Look around, it is never far See who the wound...",Country,195
541716,We started arguing on the onramp Of Interstate...,Country,283


In [4]:
print(data["words"].describe().round(1).to_string())

count    541718.0
mean        246.7
std         183.2
min           1.0
25%         138.0
50%         206.0
75%         306.0
max       16511.0


There is at least one song with just one word in the lyrics field and anotger with 16 thousand...

In [5]:
# printing the ones with just 1 word and the longest one
for _, row in data.sort_values("words").iloc[[0, 1, 2, -1]].iterrows():
    print(f"{row.words:4d} words | {row.genre}: {row.Lyric[:200]}")

   1 words | Metal: Youreokiwillbei
   1 words | Rap: Httpssoundcloud.comjlawless1murderinaminutethrowbackbobftchrisbrownfreestyle?injlawless1setsfreshfreestyles
   1 words | Rap: Httprap.genius.com520004EminemwhenimgoneJustknowthatimlookingdownonyousmilingandididntfeelathingsobabydontfeelnopainjustsmileback
16511 words | Rap: CHAPTER FOUR A FRESH LIGHT UPON ROBERT DE SAINTLOUP Oh, it is unheardof, said my mother. Listen, at my age, one has ceased to be astonished at anything, but I assure you that there could be nothing mo


Obs: Another problem arises: there are lyrics containing / that are just a link (http)

In [6]:
# check 10 and 90th percentile
print(data["words"].quantile(0.1).round(1))
print(data["words"].quantile(0.95).round(1))

91.0
573.0


According to the internet: "Most typical pop songs have around 200 to 300 words, while rap tracks often reach 500 or more." So the numbers above seem like a perfectly reasonable cut off.

In [7]:
# printing one example of each: they look allright
print(data[data["words"] == data["words"].quantile(0.1).round(1)].iloc[0]["Lyric"])
print(data[data["words"] == data["words"].quantile(0.95).round(1)].iloc[0]["Lyric"])

Well listen to me now Try to understand In this holy world Where I am trapped Im the oldest And I realize About the pact we made Where I am sick and lost I hate your wishes I want your pain All my life with you I made my own hell No return I wont be with you forever No return Anymore! Keep out of my life Ill kill you next time Forget I exist Im no one for you No return I wont be with you forever No return Anymore!
It is your lifes eternal dream To run and enter the machine A second life thats hidden from the world You take the power that it gives To realize how you want to live And build the world as it needs to be You will know her holy name Shell fulfil your frozen dream As she draws you in Beautiful child of this perfect world An icon of heavenly sin The moments of your life for sale They see the image of themselves It gives them all a chance to live as you do You draw them in to your own world And in return you come to theirs You build a planet from your hearts and hands They can s

In [8]:
# dropping all the records outside of the 10th and 90th percentile range, which we consider outliers
data = data[(data["words"] >= 91) & (data["words"] <= 573)]

In [9]:
data.shape # since it was such a large dataset, we can afford the choice to drop the 'suspicious' records

(461056, 3)

In [10]:
data['genre'].value_counts() # still fine. observation: mostly rap has been dropped

genre
Country    102091
Rock        96499
Pop         92650
Metal       87753
Rap         82063
Name: count, dtype: int64

### LYRICS WITH LINKS OR BRACKETS

In [11]:
# looking for lyrics that contain words that begin with htt
data[data['Lyric'].str.contains(r"(?i)\bhtt", na=False)].sort_values(by='words')

,Lyric,genre,words
22538,Produced by Miitri Aaltonen Unholy Recorded an...,Metal,91
243840,Cowboys in focus that time cut I kind of far T...,Rap,91
127711,Microdermabrasion or Alpha hydroxy acid If you...,Rock,91
327009,And texture however feel make these for his co...,Pop,92
235206,From WorkBook On HISTORY TASTE THIS SHIT Yeah!...,Rap,92
...,...,...,...
219841,Lyrics ScammaD Whisper Hook Livin on a high an...,Rap,560
215694,Intro Russel Rhymes Xa bra.. Xa... Is that Cho...,Rap,560
200653,Young rap artist bred by wonder and fascinatio...,Rap,563
207019,The Infamous Instrumental httpswww.youtube.com...,Rap,569


These are clearly not song lyrics, let's drop also these records.

In [12]:
data=data[~data['Lyric'].str.contains(r"(?i)\bhtt", na=False)].reset_index(drop=True)
data

,Lyric,genre,words
0,Feels like Im covered in lies so turn off the ...,Metal,179
1,"Works of art, painted black Magniloquent, blee...",Metal,121
2,Into the cage like an animal You must survive ...,Metal,140
3,Paralysed in pleasure I hear you call Lost my ...,Metal,178
4,Open my eyes full of energy As I feel my tempe...,Metal,231
...,...,...,...
460658,[Verse 1] When the sun sinks down and dreams s...,Country,249
460659,I watched from the window as she slipped from ...,Country,203
460660,"Look around, it is never far See who the wound...",Country,195
460661,We started arguing on the onramp Of Interstate...,Country,283


In [13]:
# look for lyrics that contain the character "[" or "]"
data[data['Lyric'].str.contains(r"[\[\]]", na=False)].sort_values(by='words')

,Lyric,genre,words
366864,[Verse 1] I found you deep Inside my body You ...,Country,91
283498,[Verse 1] Its too late to tell The damage done...,Pop,91
155118,"[Intro] Well, only got an hour of daylight lef...",Rock,91
430430,"Day in, day out I am always alone But I can fe...",Rock,91
353671,"[Intro] Animal Noises, Sniffing, and Slurping ...",Country,91
...,...,...,...
234245,"[Verse] Yeah, aye who are you? Why should we a...",Rap,573
224993,[Verse 1 TheoB] Turnt than a motherfucker Boug...,Rap,573
226408,"[Verse Kainiac] To me, Everyday feels like its...",Rap,573
221380,"[Syke talking] Yeah, this goes out To all my l...",Rap,573


These ARE song lyrics, but the presence of the words in [] could bias the future classification / topic modelling, so we keep the records but delete the words.

In [14]:
# For each lyrics, remove the words inside "[]" and the brackets themselves
data['Lyric'] = data['Lyric'].str.replace(r"\[.*?\]", "", regex=True)

In [15]:
data.iloc[366864]["Lyric"] # it worked

' I found you deep Inside my body You carry me When aid I need  Ohhhhhh youre my vampiric cosmic creature  I would feed you The blood from my nose bleeds I would let you fly free Cause I know you would come back to me  Ohhhhhh youre my vampiric cosmic creature  My nose bleeds A lot cause of stress But I know Youd never make me pull my teeth out and sew them on my dress  Ohhhhhh youre my vampiric cosmic creature'

### DETECTING LYRICS LANGUAGE

To adress several songs we have seen in a language that is not english. For this, we will use an already existing model from fasttext.

In [17]:
# run just once
#import urllib.request, os
#os.makedirs("models", exist_ok=True)
#urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", "models/lid.176.bin")

In [18]:
model = fasttext.load_model("models/lid.176.bin")

In [19]:
def detect_language(text):
    if not isinstance(text, str) or not text.strip():
        return None, None

    # Convert multiline lyrics into one line.
    text = " ".join(text.split())

    labels, scores = model.predict(text, k=1)

    return (
        labels[0].replace("__label__", ""),
        float(scores[0])
    )

In [20]:
data[["language", "language_score"]] = pd.DataFrame(
     data["Lyric"].map(detect_language).tolist(),
     index=data.index
 )

In [21]:
data["language"].value_counts(dropna=False)

language
en     460661
NaN         2
Name: count, dtype: int64

In [ ]:
data[data["language"]=='ja']

,Lyric,genre,words,language,language_score
118,IN A LAND OF DARKNESS ONLY WAR MAKES THEM LIVE...,Metal,153,ja,0.359576
340,SITTIN ON THE GUILT STEP LOVE AND HATE REGRET ...,Metal,90,ja,0.421225
524,SO SMALL IN MY GREATNESS I AM THE ONE WHO NEVE...,Metal,141,ja,0.364462
526,SUPREME LORDS THAT PRECEDE THEM ALL YOUVE BROU...,Metal,152,ja,0.329986
813,CENTURIES OF BLOOD AND WAR THE WORLD IS ON ITS...,Metal,165,ja,0.331328
...,...,...,...,...,...
507780,COLD BLOODED ICE IN MY VEINS ICE IN MY VEINS S...,Metal,235,ja,0.429700
508344,THIS IS THE NECESSARY INTERLUDE YOU STRIVE NOT...,Metal,182,ja,0.377823
508914,SILVERY MOONBEAMS ITS A TIME FOR MOONSET AND H...,Metal,77,ja,0.415951
509101,SEE THE BODIES DRENCHED IN POOLS OF BLOOD ALL ...,Metal,105,ja,0.314478


In [ ]:
data[['language']][data["language_score"]>=0.9].value_counts(dropna=False)

language
en          329593
de              24
es              21
it               5
af               1
fr               1
pt               1
Name: count, dtype: int64

In [ ]:
data[(data["language_score"]>=0.9)&(data["language"]=='en')]

,Lyric,genre,words,language,language_score
1,Feels like Im covered in lies so turn off the ...,Metal,179,en,0.907360
5,"Unseen, cheap, faithful weapon Buried devils a...",Metal,28,en,0.905763
6,Open my eyes full of energy As I feel my tempe...,Metal,231,en,0.920990
8,I toss and turn and lie awake...it torments me...,Metal,245,en,0.971554
9,"Two hundred years, roaming the world forgettin...",Metal,127,en,0.939124
...,...,...,...,...,...
549993,"I, I can see, that you got your sights on him ...",country,296,en,0.939903
549994,Today the pain we feel is cuttin deeper The fa...,country,172,en,0.901360
549996,I watched from the window as she slipped from ...,country,203,en,0.946845
549997,"Look around, it is never far See who the wound...",country,195,en,0.965644


In [ ]:
df_english = data[(data["language_score"]>=0.9)&(data["language"]=='en')].copy().reset_index(drop=True)

In [62]:
df_english

,Lyric,genre,words,language,language_score
0,Feels like Im covered in lies so turn off the ...,Metal,179,en,0.907360
1,"Unseen, cheap, faithful weapon Buried devils a...",Metal,28,en,0.905763
2,Open my eyes full of energy As I feel my tempe...,Metal,231,en,0.920990
3,I toss and turn and lie awake...it torments me...,Metal,245,en,0.971554
4,"Two hundred years, roaming the world forgettin...",Metal,127,en,0.939124
...,...,...,...,...,...
329588,"I, I can see, that you got your sights on him ...",country,296,en,0.939903
329589,Today the pain we feel is cuttin deeper The fa...,country,172,en,0.901360
329590,I watched from the window as she slipped from ...,country,203,en,0.946845
329591,"Look around, it is never far See who the wound...",country,195,en,0.965644
